In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

import plotly.express as px

import scanpy as sc
import scipy.sparse as sp

import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm

from movies import get_movie_ratings

In [4]:
users_df = pd.read_pickle('users_df_complete.pkl')
users_df = users_df.replace(0, np.nan)
users_df = (users_df - 5.5) / 4.5

In [8]:
def recommend_me(user, n_movies = 10):
    global users_df
    try:
        target = users_df.loc[user].to_numpy()
    except KeyError:
        print('collecting user...')
        temp = get_movie_ratings(user)
        temp_dict = {d['movie']: d['rating'] for d in temp}
        temp_df = pd.DataFrame(temp_dict, index = [user])
        temp_df = (temp_df - 5.5) / 4.5
        
        users_df = pd.concat([users_df, temp_df])

        target = users_df.loc[user].to_numpy()

    cos_sims = []
    for u, row in tqdm(users_df.iterrows(), desc = 'similarity calculation...'):
        row_arr = row.to_numpy()

        # mask NaNs (only keep overlapping indices)
        mask = ~np.isnan(target) & ~np.isnan(row_arr)
        # if there are less than 5 movies in common the similarity doesn't mean much
        if mask.sum() < 5:
            cos_sim = np.nan
        else:
            dot = np.dot(target[mask], row_arr[mask])
            cos_sim = dot / (np.linalg.norm(target[mask]) * np.linalg.norm(row_arr[mask]))
        
        cos_sims.append((u, cos_sim))

    similarity_df = pd.DataFrame(cos_sims, columns=['user', 'cos_sim'])
    similarity_df = similarity_df.loc[similarity_df['user'] != user]

    print(f'your Letterboxd buddy is {similarity_df.sort_values(by='cos_sim', ascending=False).iloc[0]['user']}')

    na_users = similarity_df.loc[similarity_df['cos_sim'].isna()]['user'].values

    others_df = users_df.drop(index=user)
    others_df = others_df.drop(index=na_users)

    # Align similarities with others
    weights = similarity_df.set_index('user').loc[others_df.index, 'cos_sim']

    # Compute weighted average per column (movies)
    # (row-wise multiply each user’s ratings by its similarity weight)
    weighted_sum = (others_df.T * weights).T.sum(axis=0)
    sum_weights = weights.sum()

    weighted_avg = weighted_sum / sum_weights

    mask_unrated = users_df.loc[user].isna()
    recommendations = weighted_avg[mask_unrated]*4.5 + 5.5

    if n_movies > len(recommendations):
        n_movies = len(recommendations)
    
    return recommendations.sort_values(ascending=False).head(n_movies)

In [9]:
recommend_me('enesidemo')

similarity calculation...: 1857it [00:16, 112.48it/s]


your Letterboxd buddy is calverdos


the-shining                 8.591516
oppenheimer-2023            8.527749
the-dark-knight             8.512292
la-la-land                  8.491279
get-out-2017                8.477969
dune-part-two               8.451588
mad-max-fury-road           8.441805
the-silence-of-the-lambs    8.416640
se7en                       8.390317
interstellar                8.389427
dtype: float64